In [1]:
import numpy as np
import pandas as pd
import yfinance as yf

# 샘플 테스트 유니버스 (실제 본선에서는 QuantConnect 전체 유니버스 / Jimmy 후보군 사용)
tickers = ["AAPL", "MSFT", "NVDA", "JPM", "XOM", "JNJ", "PG", "HD", "UNH", "INTC", "F", "BA", "T", "DIS", "COST"]

rows = []
for t in tickers:
    try:
        tk = yf.Ticker(t)
        inc = tk.financials
        bs = tk.balance_sheet
        cf = tk.cashflow
        
        if inc.empty or bs.empty or cf.empty:
            continue
            
        # 가장 최근 연간 재무제표 기준 지표 추출
        gp = inc.loc["Gross Profit"].iloc[0] if "Gross Profit" in inc.index else np.nan
        rev = inc.loc["Total Revenue"].iloc[0] if "Total Revenue" in inc.index else np.nan
        ni = inc.loc["Net Income"].iloc[0] if "Net Income" in inc.index else np.nan
        assets = bs.loc["Total Assets"].iloc[0] if "Total Assets" in bs.index else np.nan
        equity = bs.loc["Stockholders Equity"].iloc[0] if "Stockholders Equity" in bs.index else np.nan
        cfo = cf.loc["Operating Cash Flow"].iloc[0] if "Operating Cash Flow" in cf.index else np.nan
        
        gpa = gp / assets
        roe = ni / equity
        roa = ni / assets
        cfoa = cfo / assets
        gm = gp / rev
        # Accruals = (Net Income - Operating Cash Flow) / Total Assets (낮을수록 이익의 질이 높음)
        accruals = (ni - cfo) / assets
        
        rows.append({
            "Ticker": t, "GPA": gpa, "ROE": roe, "ROA": roa,
            "CFOA": cfoa, "GrossMargin": gm, "Accruals": accruals
        })
    except Exception:
        pass

q_df = pd.DataFrame(rows).dropna().set_index("Ticker")

# 횡단면(Cross-Sectional) Z-Score 계산 (Accruals는 부호 반전)
z_df = pd.DataFrame(index=q_df.index)
for col in ["GPA", "ROE", "ROA", "CFOA", "GrossMargin"]:
    z_df[f"z_{col}"] = (q_df[col] - q_df[col].mean()) / q_df[col].std()
z_df["z_Accruals"] = -(q_df["Accruals"] - q_df["Accruals"].mean()) / q_df["Accruals"].std()

# Composite Quality Z-Score (6개 지표 동일 가중 평균)
q_df["Quality_Z"] = z_df.mean(axis=1)
q_df = q_df.sort_values("Quality_Z", ascending=False)

print("=== Composite Quality Z-Score Prototype (Sample Universe) ===")
print(q_df[["GPA", "ROE", "CFOA", "GrossMargin", "Accruals", "Quality_Z"]].round(3).to_string())

=== Composite Quality Z-Score Prototype (Sample Universe) ===
          GPA    ROE   CFOA  GrossMargin  Accruals  Quality_Z
Ticker                                                       
NVDA    0.742  0.763  0.497        0.711     0.084      1.226
AAPL    0.577  1.179  0.359        0.488    -0.032      1.107
MSFT    0.297  0.302  0.241        0.679    -0.065      0.476
HD      0.522  1.105  0.155        0.333    -0.021      0.423
PG      0.345  0.297  0.155        0.502    -0.028      0.086
COST    0.437  0.258  0.178        0.128    -0.074      0.058
JNJ     0.321  0.329  0.123        0.679     0.011      0.025
T       0.178  0.199  0.096        0.596    -0.044     -0.113
DIS     0.189  0.089  0.093        0.380    -0.045     -0.301
XOM     0.159  0.111  0.116        0.220    -0.052     -0.359
UNH     0.268  0.128  0.064        0.185    -0.025     -0.485
INTC    0.087 -0.002  0.046        0.348    -0.047     -0.546
F       0.006 -0.228  0.074        0.009    -0.102     -0.727
BA      